# 🏆 Notebook 27 — Capstone B: AI Customer-Feedback Assistant

> **Module:** Capstones · **Estimated time:** 90–120 min · **Difficulty:** Advanced · **Prerequisites:** Modules 1–6, especially NB 18–22 (AI engineering) and NB 23–24 (production).

The second capstone is the *engineering* twin of NB 26. Where NB 26 turned numerical data into a manager's PDF, this notebook turns *free-form text* into actionable structured output — and wraps the whole thing in the production discipline you learned in Module 6.

**You're going to build an AI Customer-Feedback Assistant** that:

1. Ingests raw customer feedback (the kind a support inbox produces).
2. Classifies sentiment + topic with structured LLM output.
3. Routes high-priority items by combining the classification with a small rules engine.
4. Grounds answers in your *own* knowledge base via RAG.
5. Produces a daily report — runnable on a schedule, with cost tracking and an eval guard.

By the end you'll have ~250 lines of self-contained Python that demonstrate every AI-engineering pattern in the course. This is the artefact you point to in an interview when asked "what AI project have you built?".

## 🎯 What you will accomplish

1. Combine **prompting**, **RAG**, **validation**, **tracing**, and **scheduling** in one end-to-end pipeline.
2. Build a **golden eval set** and a **regression check** for the classifier.
3. Compute a **cost / latency / accuracy dashboard** from the trace log.
4. Write a 5-bullet **executive summary** of yesterday's feedback for a product manager.

## ✅ Prerequisites

Notebooks 1–24 (NB 9 and 25 are intentionally absent). Especially Modules 5 (AI engineering) and 6 (production).

## 📐 The architecture you'll implement

```
   ┌─────────────────────────────────────────────────────────────────┐
   │                                                                  │
   │   Raw feedback (DataFrame)                                       │
   │            │                                                     │
   │            ▼                                                     │
   │   ┌──────────────────┐    ┌──────────────────┐                  │
   │   │  Classify        │───►│  Validate (JSON  │                  │
   │   │  (LLM, JSON)     │    │  schema)         │                  │
   │   └──────────────────┘    └────────┬─────────┘                  │
   │                                     │                            │
   │            ┌────────────────────────┴──┐                         │
   │            ▼                            ▼                        │
   │   ┌────────────────┐         ┌──────────────────┐                │
   │   │ Rules engine   │         │  RAG: ground     │                │
   │   │ (priority)     │         │  answer in docs   │                │
   │   └───────┬────────┘         └────────┬─────────┘                │
   │           │                            │                          │
   │           └──────────────┬─────────────┘                         │
   │                          ▼                                       │
   │                ┌──────────────────────┐                          │
   │                │  Aggregate + report  │                          │
   │                └──────────────────────┘                          │
   │                          │                                       │
   │            ┌─────────────┴─────────────┐                         │
   │            ▼                            ▼                        │
   │   Daily executive summary       Cost / latency / eval trace      │
   │                                                                  │
   └─────────────────────────────────────────────────────────────────┘
```

## 1. Setup — the shared mock infrastructure

In [ ]:
import json
import re
import time
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})


class MockLLM:
    """Deterministic offline mock; same contract as a real chat-completions API."""
    POSITIVE = {"love", "great", "amazing", "excellent", "happy", "fantastic",
                "perfect", "delight", "smooth", "fast", "thank", "good", "fantastic"}
    NEGATIVE = {"hate", "terrible", "awful", "bad", "slow", "broken", "bug",
                "crash", "angry", "frustrating", "refund", "cancel", "worst"}
    BILLING  = {"refund", "invoice", "charge", "bill", "subscription", "renew", "payment", "price"}
    TECH     = {"error", "crash", "bug", "broken", "login", "password", "loading", "freeze", "stuck"}
    FEATURE  = {"feature", "request", "missing", "wish", "add", "support"}

    def _score(self, text, vocab):
        return sum(1 for w in re.findall(r"[a-z']+", text.lower()) if w in vocab)

    def chat(self, messages, model="mock-mini", temperature=0.0):
        sys_msg  = next((m["content"] for m in messages if m["role"] == "system"), "").lower()
        user_msg = [m["content"] for m in messages if m["role"] == "user"][-1]

        if "json" in sys_msg and ("sentiment" in sys_msg or "topic" in sys_msg):
            pos, neg = self._score(user_msg, self.POSITIVE), self._score(user_msg, self.NEGATIVE)
            sentiment = "positive" if pos > neg else "negative" if neg > pos else "neutral"
            scores = {"billing": self._score(user_msg, self.BILLING),
                      "tech":    self._score(user_msg, self.TECH),
                      "feature": self._score(user_msg, self.FEATURE)}
            topic = max(scores, key=scores.get) if max(scores.values()) > 0 else "other"
            out = json.dumps({"sentiment": sentiment, "topic": topic})
        elif "answer the user's question" in sys_msg or "qa" in sys_msg:
            # Echo a short context-based answer (mock)
            out = "Based on the provided context, " + user_msg.split("Question:")[-1].strip()[:120]
        else:
            out = "..."

        tokens_in  = sum(len(m["content"]) for m in messages) // 4
        tokens_out = len(out) // 4
        return {"text": out, "model": model,
                "tokens_in": tokens_in, "tokens_out": tokens_out,
                "latency_s": 0.001 + 0.0001 * tokens_out}


llm = MockLLM()
print("MockLLM ready ✅")


> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. Once you want real intelligence in the answers, swap one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else.
> See the [LLM Providers Guide](./A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


## 2. The data — incoming feedback and a small knowledge base

In [ ]:
# 50 fictional feedback messages (the kind a support inbox sees overnight)
FEEDBACK = [
    "Love the new dashboard! It's so much faster.",
    "The app crashed three times today, I'm so frustrated.",
    "Could you please add CSV export? I really need it for my reports.",
    "My invoice doesn't match my plan, please refund the difference of $45.",
    "Everything works fine, no complaints.",
    "Login is broken after the latest update.",
    "Fantastic support team — they resolved my issue in minutes. Thank you!",
    "Why is the renewal price so high? I'm considering cancelling.",
    "The new feature is great but loading is really slow on mobile.",
    "Please add Markdown support in the editor.",
    "Charged twice for my subscription last month.",
    "Dark mode would be amazing — please prioritise.",
    "500 error on file upload. Help.",
    "Smooth experience overall, would recommend.",
    "Fantastic update — keep up the good work!",
    "I'm cancelling, this is unusable.",
    "Password reset email never arrives.",
    "Bill is wrong again, third time this quarter.",
    "Great product. Five stars.",
    "Mobile app freezes whenever I try to upload images.",
    "Refund requested last week — still no update.",
    "Could you support SSO with Okta?",
    "Latest patch broke the export feature.",
    "Pricing page is confusing.",
    "Loving the new automations!",
    "Stuck at the loading screen, can't access my data.",
    "Add an Indonesian translation please.",
    "Charged for a plan I never signed up to.",
    "Wonderful UI, very intuitive.",
    "Crash on Windows 11, every time.",
    "Customer service was rude on the phone.",
    "I want a full refund. This is unacceptable.",
    "The API documentation is excellent, thank you.",
    "How do I cancel? The button is hidden.",
    "Cool new dashboard! Lovely details.",
    "Can you add a CSV export for the analytics page?",
    "The mobile version is unusable lately.",
    "Got billed twice, please fix.",
    "Slack integration would be a game changer.",
    "Found a major security issue, contacting you privately.",
    "Smooth as silk, the new release works perfectly.",
    "The webhook endpoint returns 500 randomly.",
    "Annual discount doesn't apply on my account.",
    "Love the speed improvements. Top job!",
    "Cancellation page is confusing.",
    "Could you support a Zapier integration?",
    "Login keeps logging me out after 5 minutes.",
    "Pricing is too high for what we get.",
    "Excellent customer service, would recommend.",
    "The new mobile app is much better.",
]
print(f"Loaded {len(FEEDBACK)} feedback messages.")


In [ ]:
# Knowledge base — answers to common questions, used in the RAG step
KB = [
    ("billing.refunds",
     "Refunds are issued within 5 business days. Request from the Billing page within 30 days of purchase."),
    ("billing.subscriptions",
     "Subscriptions renew automatically. Cancel from Settings → Billing → Cancel subscription. You retain access until period end."),
    ("billing.pricing",
     "Three tiers: Free, Pro $19/mo, Enterprise on request. Annual billing gives a 15% discount."),
    ("tech.password",
     "Use 'Forgot password' on the sign-in page. Reset emails arrive within 5 minutes; check spam."),
    ("tech.errors",
     "A 500 error usually indicates a temporary backend issue. Wait 60 seconds and retry. Contact support with the trace ID if persistent."),
    ("feature.exports",
     "CSV and JSON exports are on Pro plans. XLSX is on the roadmap."),
    ("feature.integrations",
     "Slack, Microsoft Teams, and Zapier are supported. Okta SSO is Enterprise-only."),
]
print(f"Knowledge base: {len(KB)} documents.")


## 3. Step 1 of the pipeline — classify with structured output

In [ ]:
CLASSIFY_SYSTEM = (
    "You analyse customer messages. "
    "Return JSON with keys 'sentiment' (positive/neutral/negative) and "
    "'topic' (billing/tech/feature/other). Reply with the JSON only."
)

def classify(text: str) -> dict:
    """Run a single feedback through the classifier; return a typed dict."""
    r = llm.chat(messages=[
        {"role": "system", "content": CLASSIFY_SYSTEM},
        {"role": "user",   "content": text},
    ])
    try:
        labels = json.loads(r["text"])
    except json.JSONDecodeError:
        labels = {"sentiment": "unknown", "topic": "unknown"}
    return {
        "sentiment":  labels.get("sentiment", "unknown"),
        "topic":      labels.get("topic", "unknown"),
        "tokens_in":  r["tokens_in"],
        "tokens_out": r["tokens_out"],
        "latency_s":  r["latency_s"],
    }


# Quick test
print(classify("My invoice is wrong, please refund."))


## 4. Step 2 — validate the structured output

The model is fast but not infallible. Always validate before pushing data downstream.

In [ ]:
VALID_SENTIMENTS = {"positive", "neutral", "negative"}
VALID_TOPICS     = {"billing", "tech", "feature", "other"}

def validate(record: dict) -> list[str]:
    errs = []
    if record.get("sentiment") not in VALID_SENTIMENTS:
        errs.append(f"bad sentiment: {record.get('sentiment')!r}")
    if record.get("topic") not in VALID_TOPICS:
        errs.append(f"bad topic: {record.get('topic')!r}")
    return errs


# Test
print(validate({"sentiment": "positive", "topic": "feature"}))         # []
print(validate({"sentiment": "happy",    "topic": "tech"}))            # ["bad sentiment: 'happy'"]


## 5. Step 3 — the priority rules engine

A rules engine on top of the classification. This is where business logic lives. Keep it *separate* from the model so it's testable and reviewable on its own.

In [ ]:
def priority(text: str, sentiment: str, topic: str, is_vip: bool = False) -> str:
    """Return one of: 'p0_escalate', 'p1_negative', 'p2_route', 'p3_acknowledge'."""
    lower = text.lower()
    if any(w in lower for w in ("urgent", "security", "lawyer", "escalate", "asap")):
        return "p0_escalate"
    if is_vip and sentiment == "negative":
        return "p0_escalate"
    if sentiment == "negative" and topic in {"billing", "tech"}:
        return "p1_negative"
    if topic == "feature":
        return "p2_route"          # collect for the product team
    return "p3_acknowledge"


# Test priorities
for txt in [
    "My invoice is wrong, please refund.",
    "Add a Zapier integration please.",
    "Found a security issue, contacting privately.",
    "Love the new dashboard.",
]:
    cls = classify(txt)
    print(f"{priority(txt, cls['sentiment'], cls['topic']):<16}  ← {txt!r}")


## 6. Step 4 — RAG for actually answering the question

For items that ask a concrete question (refund, password, integrations), pull the answer from the knowledge base.

In [ ]:
def keyword_score(query, doc_text):
    q = set(re.findall(r"[a-z']+", query.lower()))
    d = set(re.findall(r"[a-z']+", doc_text.lower()))
    return len(q & d)

def retrieve(query, k=2):
    scored = sorted([(keyword_score(query, t), n, t) for n, t in KB], reverse=True)
    return [(n, t) for s, n, t in scored[:k] if s > 0]

def rag_answer(question: str) -> dict:
    snippets = retrieve(question, k=2)
    if not snippets:
        return {"answer": None, "sources": [], "tokens_in": 0, "tokens_out": 0, "latency_s": 0}
    context = "\n\n".join(f"[{n}] {t}" for n, t in snippets)
    r = llm.chat(messages=[
        {"role": "system", "content":
            "Answer the user's question using ONLY the context provided. If the context does not contain the answer, say so."},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])
    return {"answer": r["text"], "sources": [n for n, _ in snippets],
            "tokens_in": r["tokens_in"], "tokens_out": r["tokens_out"],
            "latency_s": r["latency_s"]}


# Quick test
result = rag_answer("How do I cancel my subscription?")
print("Answer :", result["answer"])
print("Sources:", result["sources"])


## 7. Step 5 — the full pipeline + tracing

Every step records its own row of trace data so we can debug, evaluate, and report cost.

In [ ]:
@dataclass
class Outcome:
    request_id:  str
    text:        str
    sentiment:   str
    topic:       str
    priority:    str
    rag_answer:  str | None
    sources:     list
    tokens_in:   int
    tokens_out:  int
    latency_s:   float
    validation_errors: list


def run_pipeline(feedback: list[str], is_vip_fn=lambda t: False) -> list[Outcome]:
    """Process every message end-to-end and return a list of typed outcomes."""
    outcomes = []
    for text in feedback:
        request_id = hashlib.sha1(f"{time.time()}::{text}".encode()).hexdigest()[:10]

        # 1) Classify
        cls = classify(text)
        errs = validate(cls)

        # 2) Priority
        prio = priority(text, cls["sentiment"], cls["topic"], is_vip=is_vip_fn(text))

        # 3) RAG only for p1/p2/p3 (skip p0 — escalation goes to a human anyway)
        rag = {"answer": None, "sources": [], "tokens_in": 0, "tokens_out": 0, "latency_s": 0}
        if prio in {"p1_negative", "p2_route", "p3_acknowledge"} and not errs:
            rag = rag_answer(text)

        outcomes.append(Outcome(
            request_id=request_id, text=text,
            sentiment=cls["sentiment"], topic=cls["topic"], priority=prio,
            rag_answer=rag["answer"], sources=rag["sources"],
            tokens_in =cls["tokens_in"]  + rag["tokens_in"],
            tokens_out=cls["tokens_out"] + rag["tokens_out"],
            latency_s =cls["latency_s"]  + rag["latency_s"],
            validation_errors=errs,
        ))
    return outcomes


outcomes = run_pipeline(FEEDBACK)
trace_df = pd.DataFrame([asdict(o) for o in outcomes])
print(f"Processed {len(trace_df)} messages.")
print(trace_df[["sentiment", "topic", "priority", "tokens_in", "tokens_out"]].head(8))


## 8. The KPI dashboard

In [ ]:
# Some quick numbers
n = len(trace_df)
priority_counts  = trace_df["priority"].value_counts().reindex(
    ["p0_escalate","p1_negative","p2_route","p3_acknowledge"], fill_value=0)
sentiment_counts = trace_df["sentiment"].value_counts()
topic_counts     = trace_df["topic"].value_counts()

# Costs (using a typical input/output rate)
PRICE_IN_PER_1K  = 0.0006
PRICE_OUT_PER_1K = 0.0024
trace_df["cost_usd"] = (trace_df["tokens_in"]  / 1000 * PRICE_IN_PER_1K +
                        trace_df["tokens_out"] / 1000 * PRICE_OUT_PER_1K)

print(f"📊 AI Feedback Assistant — Run Summary")
print("-" * 60)
print(f"Messages processed : {n}")
print(f"Validation errors  : {(trace_df['validation_errors'].str.len() > 0).sum()}")
print(f"Total cost         : ${trace_df['cost_usd'].sum():.4f}")
print(f"Mean cost / msg    : ${trace_df['cost_usd'].mean():.5f}")
print(f"Mean latency       : {trace_df['latency_s'].mean()*1000:.1f} ms")
print(f"p95 latency        : {trace_df['latency_s'].quantile(0.95)*1000:.1f} ms")
print("\nPriority mix:")
for p, c in priority_counts.items():
    print(f"  {p:<18}: {c:>3}  ({c/n:.0%})")
print("\nSentiment mix:")
for s, c in sentiment_counts.items():
    print(f"  {s:<10}: {c:>3}  ({c/n:.0%})")
print("\nTopic mix:")
for t, c in topic_counts.items():
    print(f"  {t:<10}: {c:>3}  ({c/n:.0%})")


In [ ]:
# Visual dashboard — four panels
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
fig.suptitle("AI Feedback Assistant — daily dashboard",
             fontsize=15, fontweight="bold")

# (0,0) Priority bar
ax = axes[0, 0]
colors_prio = ["#C44E52", "#DD8452", "#CCB974", "#55A467"]
ax.bar(priority_counts.index, priority_counts.values,
       color=colors_prio, edgecolor="black")
for x, v in zip(priority_counts.index, priority_counts.values):
    ax.text(x, v, str(v), ha="center", va="bottom", fontsize=10)
ax.set_title("Priority mix"); ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=15)

# (0,1) Sentiment × Topic heatmap
ax = axes[0, 1]
cross = pd.crosstab(trace_df["topic"], trace_df["sentiment"]).reindex(
    columns=["positive","neutral","negative"], fill_value=0)
im = ax.imshow(cross.values, cmap="Blues")
ax.set_xticks(range(cross.shape[1]), cross.columns)
ax.set_yticks(range(cross.shape[0]), cross.index)
for i in range(cross.shape[0]):
    for j in range(cross.shape[1]):
        ax.text(j, i, cross.values[i, j], ha="center", va="center",
                color="white" if cross.values[i, j] > cross.values.max()/2 else "black",
                fontsize=11)
ax.set_title("Sentiment × Topic")
fig.colorbar(im, ax=ax, label="count")

# (1,0) Cumulative cost
ax = axes[1, 0]
ax.plot(range(len(trace_df)), trace_df["cost_usd"].cumsum() * 100,
        lw=2, color="#4C72B0")
ax.set_xlabel("Message #")
ax.set_ylabel("Cumulative cost (¢)")
ax.set_title("Cumulative spend through the run")

# (1,1) Latency distribution
ax = axes[1, 1]
ax.hist(trace_df["latency_s"] * 1000, bins=15, color="#8172B2",
        edgecolor="black", alpha=0.85)
ax.axvline(trace_df["latency_s"].quantile(0.95) * 1000, color="red", ls="--",
            label=f"p95 = {trace_df['latency_s'].quantile(0.95)*1000:.1f} ms")
ax.set_xlabel("ms"); ax.set_ylabel("count")
ax.set_title("Latency distribution"); ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 9. The executive summary — what a PM actually wants to read

The artefact of this whole pipeline isn't the dashboard. It's a *paragraph* that summarises last night's feedback in language a manager can act on.

In [ ]:
def write_executive_summary(trace_df: pd.DataFrame) -> str:
    n = len(trace_df)
    p0 = (trace_df["priority"] == "p0_escalate").sum()
    p1 = (trace_df["priority"] == "p1_negative").sum()
    neg_billing = ((trace_df["sentiment"] == "negative") &
                   (trace_df["topic"] == "billing")).sum()
    top_topic = trace_df["topic"].value_counts().idxmax()
    pos_rate  = (trace_df["sentiment"] == "positive").mean()

    bullets = [
        f"📊 Processed {n} messages overnight at ${trace_df['cost_usd'].sum():.4f} total LLM spend.",
        f"🚨 {p0} P0 escalations require human review immediately.",
        f"⚠️  {p1} negative-sentiment messages flagged in billing/tech for first-response.",
        f"💸 {neg_billing} billing-related complaints — investigate any refund-process spike.",
        f"📈 Positive feedback rate: {pos_rate:.0%}.  Top topic: '{top_topic}'.",
    ]
    return "\n".join(bullets)


summary = write_executive_summary(trace_df)
print(summary)


## 10. The regression check — would tomorrow's prompt change still pass?

Before any prompt edit ships, we re-run the eval against a golden set. If the new prompt regresses, the deploy fails.

In [ ]:
# Same golden set you'd build for an AI feature (matches what NB 22 demonstrated)
GOLDEN = [
    ("I love the new dashboard! It's so much faster.",        "positive", "feature"),
    ("The app crashed three times today.",                     "negative", "tech"),
    ("Could you please add CSV export?",                       "neutral",  "feature"),
    ("My invoice doesn't match my plan, please refund.",       "negative", "billing"),
    ("Login is broken after the latest update.",               "negative", "tech"),
    ("Fantastic support team!",                                "positive", "other"),
    ("Why is the renewal price so high?",                      "negative", "billing"),
]

def regression_check(prompt: str, baseline_acc: float, tolerance: float = 0.10) -> bool:
    rows = []
    for text, gold_s, gold_t in GOLDEN:
        r = llm.chat(messages=[
            {"role": "system", "content": prompt},
            {"role": "user",   "content": text},
        ])
        try:
            pred = json.loads(r["text"])
        except json.JSONDecodeError:
            pred = {"sentiment": "unknown", "topic": "unknown"}
        rows.append({
            "text": text,
            "ok": pred.get("sentiment") == gold_s and pred.get("topic") == gold_t,
        })
    res = pd.DataFrame(rows)
    acc = res["ok"].mean()
    print(f"Eval accuracy: {acc:.0%}  (baseline {baseline_acc:.0%}, tolerance {tolerance:.0%})")
    if acc < baseline_acc - tolerance:
        print("❌ Regression — refusing to ship this prompt.")
        return False
    print("✅ Eval passed.")
    return True


# Run the check against the current pipeline
regression_check(CLASSIFY_SYSTEM, baseline_acc=0.70)


## 11. Wiring it all together — a single scheduled task

This is the function a cron job (or a Prefect flow) would invoke nightly. It produces the daily report file, the cost dashboard, and emits the executive summary to (mock) Slack.

In [ ]:
def daily_run(feedback: list[str], baseline_acc: float = 0.65) -> dict:
    """
    The full nightly task. Returns a status dict for the scheduler to log.
    """
    # 1) Regression guard — never ship a bad prompt
    if not regression_check(CLASSIFY_SYSTEM, baseline_acc):
        return {"status": "halted", "reason": "prompt regression"}

    # 2) Process the inbox
    outcomes = run_pipeline(feedback)
    trace_df = pd.DataFrame([asdict(o) for o in outcomes])

    # 3) Compute cost
    PRICE_IN_PER_1K, PRICE_OUT_PER_1K = 0.0006, 0.0024
    trace_df["cost_usd"] = (trace_df["tokens_in"]  / 1000 * PRICE_IN_PER_1K +
                            trace_df["tokens_out"] / 1000 * PRICE_OUT_PER_1K)

    # 4) Save the trace and the report
    out_dir = Path("/tmp/feedback_reports")
    out_dir.mkdir(exist_ok=True)
    trace_df.to_csv(out_dir / "trace.csv", index=False)
    summary = write_executive_summary(trace_df)
    (out_dir / "summary.txt").write_text(summary)

    # 5) Mock-send Slack
    print(f"\n[mock Slack send] daily-summary channel:\n{summary}\n")

    return {
        "status":           "ok",
        "n_messages":       len(trace_df),
        "cost_usd":         round(trace_df["cost_usd"].sum(), 5),
        "p0_escalations":   int((trace_df["priority"] == "p0_escalate").sum()),
        "report_dir":       str(out_dir),
    }


result = daily_run(FEEDBACK)
print(f"\nResult: {result}")


**Read what just happened.** In a single function, you:

- Guarded against shipping a regressed prompt (the eval gate).
- Classified, validated, and routed 50 free-form customer messages.
- Computed cost, latency, and accuracy for the run.
- Saved a trace to disk that any later debugging session can read.
- Produced a five-bullet executive summary you'd actually send to a PM.

This is the *engineering* version of a capstone. Schedule it nightly (NB 24), wrap it in a package (NB 23), and you've shipped real production AI.

## 🧪 Bonus exercises

### Exercise 1 — Add an English-only filter

The pipeline currently processes every message. Add a step that detects when a message is *not in English* and routes it to `p2_route` regardless of priority. Use a tiny rule: if more than 30% of characters aren't ASCII letters/spaces/punctuation, treat as non-English.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
import string
def is_likely_english(text: str) -> bool:
    allowed = set(string.ascii_letters + string.digits + string.whitespace + string.punctuation)
    ratio = sum(c in allowed for c in text) / max(1, len(text))
    return ratio >= 0.7

# Hook it into run_pipeline by checking after classify and overriding priority:
#   if not is_likely_english(text): prio = "p2_route"
print(is_likely_english("Hello, how are you?"))   # True
print(is_likely_english("こんにちは、元気ですか？"))   # False
```

In production you'd use a real language-detection library (`langdetect`, `langid`, or an LLM call). The ASCII heuristic catches most cases for free.
</details>

### Exercise 2 — Add VIP-customer routing

Add a `vip_customers` set (a small list of customer IDs or email patterns). When their feedback comes in, *any* negative sentiment should escalate to p0. Demonstrate it by adding `is_vip_fn=lambda t: "VIP" in t` to `daily_run`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def is_vip(text: str) -> bool:
    return "[VIP]" in text or "vip-customer" in text.lower()

# Then call:
outcomes_vip = run_pipeline(FEEDBACK, is_vip_fn=is_vip)
```

Try inserting `"[VIP] My invoice is wrong."` into `FEEDBACK` and verify it's routed to `p0_escalate` even though the rule for non-VIPs would mark it `p1_negative`.

In production the VIP list lives in your CRM and is loaded as configuration — never hard-coded.
</details>

### Exercise 3 — Bake the daily summary into a markdown report

Modify `daily_run` to also emit a `report.md` with:

- The summary
- A markdown table of the priority/sentiment/topic crosstabs
- A link to the saved trace.csv

This is what you'd commit to a `reports/` folder, or email as an attachment.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def render_markdown_report(trace_df: pd.DataFrame, out_dir: Path) -> Path:
    summary = write_executive_summary(trace_df)
    sentiment_crosstab = pd.crosstab(trace_df["topic"], trace_df["sentiment"]).to_markdown()
    priority_summary   = trace_df["priority"].value_counts().to_markdown()

    md = f"""# AI Feedback Assistant — Daily Report

## Executive summary

{summary}

## Priority mix

{priority_summary}

## Topic × Sentiment

{sentiment_crosstab}

## Raw trace

See [trace.csv]({out_dir}/trace.csv) for every message processed.
"""
    path = out_dir / "report.md"
    path.write_text(md)
    return path


path = render_markdown_report(trace_df, Path("/tmp/feedback_reports"))
print(f"Wrote {path}")
print((path).read_text()[:600])
```

The pattern — *trace.csv for engineers, report.md for managers* — is the single most useful artefact split in production data pipelines. Engineers debug from the CSV; managers act on the Markdown.
</details>

## 🎓 What you've built

This is the *engineering* capstone. The artefact you have at the end:

- A package-shaped function that processes 50+ messages end-to-end.
- An eval gate that prevents a bad prompt from shipping.
- A cost / latency / accuracy dashboard you can hand to a PM.
- An executive summary good enough to email.
- A trace log for any later debugging session.

Combine this with the analytical capstone (NB 26) and you can credibly tell an interviewer: *"Here's how I'd build the next AI feature your company needs."*

## 🚀 Where to go from here

A handful of natural next steps once you've shipped this:

1. **Replace `MockLLM` with a real provider** (OpenAI / Anthropic / a local model). The function signatures are identical.
2. **Swap keyword retrieval for embedding-based retrieval** (NB 19).
3. **Add tool calling** so the assistant can lookup CRM data on its own (NB 20).
4. **Add a UI** — Streamlit gets you a working demo in 30 lines.
5. **Add a feedback loop**: capture user reactions ("👍 / 👎"), feed them into the next golden-set update.

You've now closed the loop. **Code, test, ship, observe, improve.** That's the entire job.

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Daily cost cap

Add a `daily_cost_cap_usd` parameter to `daily_run`. If the total cost would exceed the cap, halt processing and emit a 'cost cap exceeded' Slack alert. Verify with a cap of $0.001.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def daily_run_capped(feedback, baseline_acc=0.50, cost_cap_usd=1.0):
    if not regression_check(CLASSIFY_SYSTEM, baseline_acc):
        return {"status": "halted", "reason": "regression"}

    processed, total_cost = [], 0.0
    for text in feedback:
        rec = run_pipeline([text])[0]
        total_cost += rec.tokens_in / 1000 * 0.0006 + rec.tokens_out / 1000 * 0.0024
        processed.append(rec)
        if total_cost > cost_cap_usd:
            send_slack_alert(f"⚠ Cost cap ${cost_cap_usd} exceeded after {len(processed)} messages.")
            break
    return {"status": "ok" if total_cost <= cost_cap_usd else "capped",
            "n_messages": len(processed),
            "cost_usd":  round(total_cost, 5)}


print(daily_run_capped(FEEDBACK, cost_cap_usd=0.001))
```

**Cost caps are the single most important LLM-production safeguard.**
Bugs are infinite-loop *expensive* when you're calling a paid API.
Every scheduled AI job should have a hard cap *and* alert.

</details>

### Stretch exercise B — VIP customers always escalate

Add a `VIP_PATTERNS` list (e.g. `['[VIP]', 'enterprise@', 'priority-1']`) and modify `priority()` so any message matching a VIP pattern is always **p0_escalate**, regardless of sentiment or topic.


<details>
<summary>💡 <b>Solution</b></summary>

```python
VIP_PATTERNS = ["[VIP]", "enterprise@", "priority-1", "vip-customer"]

def priority_with_vip(text, sentiment, topic):
    lower = text.lower()
    if any(pat.lower() in lower for pat in VIP_PATTERNS):
        return "p0_escalate"
    return priority(text, sentiment, topic)


# Verify
for txt in [
    "[VIP] My invoice is wrong",
    "My invoice is wrong",
    "Love the new feature!",
]:
    cls = classify(txt)
    print(f"{priority_with_vip(txt, cls['sentiment'], cls['topic']):<16}  ← {txt!r}")
```

**VIP overrides should live in code, not in a prompt.** Prompts
drift; code reviews catch bugs in business logic. Keep
business rules and AI logic *separated* — much easier to audit.

</details>